# 第 4 章 差分方程式: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/04-difference-equations.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
a = 0.5            # 差分方程式 x(t+1) = a·x(t) + b の係数 a（1 は不可）
b = 1.0            # 定数 b
initial = 0.0      # 初期値 x(0)
periods = 20       # 何期先まで計算するか
demand_intercept = 100  # 蜘蛛の巣: 需要 Q = 切片 − 傾き·P の切片
demand_slope = 3        # 需要の傾き
supply_intercept = 0    # 供給 Q = 切片 + 傾き·P(前期) の切片
supply_slope = 2        # 供給の傾き
initial_price = 10      # 最初の価格
cobweb_periods = 6      # 蜘蛛の巣を何期描くか

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and Path(getattr(_lib, "__file__", "") or "").resolve() != (_root / "lib" / "__init__.py").resolve():
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 経路と定常状態

In [ ]:
from lib.chapter04.difference import is_stable, linear_path, steady_state

path = linear_path(a, b, initial, periods)
print("x(1) から x(5):", [round(v, 4) for v in path[1:6]])
print("定常状態 x*:", round(steady_state(a, b), 4))
print("安定か（|a| < 1）:", is_stable(a))
print(f"x({periods}):", round(path[periods], 4))

## 図: 差分方程式の経路

In [ ]:
from lib.chapter04.plot import plot_paths

plot_paths({f"a = {a}": path}, steady_state=steady_state(a, b))

## 蜘蛛の巣モデル

In [ ]:
from lib.chapter04.difference import cobweb_coefficients

a_cobweb, b_cobweb = cobweb_coefficients(
    demand_intercept, demand_slope, supply_intercept, supply_slope
)
print(f"価格の差分方程式の係数 a = {a_cobweb:.4f}、定数 b = {b_cobweb:.4f}")
print("安定か:", is_stable(a_cobweb))

In [ ]:
from lib.chapter04.plot import plot_cobweb

plot_cobweb(
    demand_intercept,
    demand_slope,
    supply_intercept,
    supply_slope,
    initial_price,
    cobweb_periods,
)

## 試してみよう

- `a = -0.8` にすると、定常状態は 0.5556 で、x(t) は 1.0、0.2、0.84、0.328、
  0.7376 と、**定常状態の上下を往復しながら**近づきます
- `a = 1.1`、`b = -0.2` にすると、|a| > 1 なので、x(20) は −11.455 で、
  定常状態（2.0）から**遠ざかります**
- `supply_slope = 4`（供給の傾きが、需要の傾き 3 より大きい）にすると、
  蜘蛛の巣の係数は a = −1.3333 で、安定ではありません。`supply_slope = 3` なら、
  a = −1.0 で、これも安定ではありません